# 05.3 — GRU from Scratch

**Problem it solves:** LSTM has 4 gates and 4 weight matrices — complex and slow. GRU (Cho et al., 2014) achieves similar performance with 3 gates and 2 weight matrices by merging the forget and input gates.

**GRU equations:**
- **Reset gate:** r_t = σ(W_r [h_{t-1}, x_t]) — how much past to use when computing candidate
- **Update gate:** z_t = σ(W_z [h_{t-1}, x_t]) — how much to update hidden state
- **Candidate:** h̃_t = tanh(W_h [r_t ⊙ h_{t-1}, x_t])
- **Output:** h_t = (1 - z_t) ⊙ h_{t-1} + z_t ⊙ h̃_t

**Key insight:** The update gate plays the role of both forget gate and input gate. z_t→1 means forget old, take new. z_t→0 means keep old.

---

In [ ]:
import numpy as np

def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-500,500)))

class GRUCell:
    def __init__(self, input_size, hidden_size):
        H, D = hidden_size, input_size
        scale = np.sqrt(2.0/(D+H))
        
        # 3 gate weight matrices (vs 4 in LSTM)
        # r: reset, z: update, h: candidate
        self.W_r = np.random.randn(H, D+H) * scale
        self.W_z = np.random.randn(H, D+H) * scale
        self.W_h = np.random.randn(H, D+H) * scale
        self.b_r = np.zeros((H, 1))
        self.b_z = np.zeros((H, 1))
        self.b_h = np.zeros((H, 1))
        
        self.hidden_size = H
        self.input_size = D
    
    def forward(self, x, h_prev):
        xh = np.vstack([x, h_prev])  # [D+H, 1]
        
        r = sigmoid(self.W_r @ xh + self.b_r)  # reset gate
        z = sigmoid(self.W_z @ xh + self.b_z)  # update gate
        
        # Candidate uses RESET h_prev
        xrh = np.vstack([x, r * h_prev])  # selectively use past
        h_tilde = np.tanh(self.W_h @ xrh + self.b_h)
        
        # Interpolate: keep old or take new?
        # z=1: full update (forget old, take candidate)
        # z=0: keep old hidden state exactly
        h_next = (1 - z) * h_prev + z * h_tilde
        
        cache = (x, h_prev, r, z, h_tilde, xrh)
        return h_next, cache
    
    def backward(self, dh_next, cache):
        x, h_prev, r, z, h_tilde, xrh = cache
        H, D = self.hidden_size, self.input_size
        
        # Gradient through interpolation
        dh_prev_direct = dh_next * (1 - z)  # carries gradient backward
        dz = dh_next * (h_tilde - h_prev)
        dh_tilde = dh_next * z
        
        # Through tanh of candidate
        dh_tilde_raw = dh_tilde * (1 - h_tilde**2)
        dW_h = dh_tilde_raw @ xrh.T
        db_h = dh_tilde_raw
        dxrh = self.W_h.T @ dh_tilde_raw
        
        dx_from_h = dxrh[:D]
        dr_h_prev = dxrh[D:]  # gradient through r * h_prev
        
        # Gradient through reset gate
        dr = dr_h_prev * h_prev
        dh_prev_from_r = dr_h_prev * r
        
        dr_raw = dr * r * (1 - r)
        dz_raw = dz * z * (1 - z)
        
        xh = np.vstack([x, h_prev])
        dW_r = dr_raw @ xh.T
        dW_z = dz_raw @ xh.T
        db_r = dr_raw
        db_z = dz_raw
        
        dxh_r = self.W_r.T @ dr_raw
        dxh_z = self.W_z.T @ dz_raw
        
        dx = dx_from_h + dxh_r[:D] + dxh_z[:D]
        dh_prev = (dh_prev_direct + dh_prev_from_r + 
                   dxh_r[D:] + dxh_z[D:])
        
        return dx, dh_prev, dW_r, dW_z, dW_h, db_r, db_z, db_h

# Compare LSTM vs GRU
from lstm_cell_demo import LSTMCell  # if in same session

D, H = 10, 16
lstm = LSTMCell(D, H)
gru  = GRUCell(D, H)

lstm_params = lstm.W.size + lstm.b.size
gru_params  = sum(p.size for p in [gru.W_r, gru.W_z, gru.W_h, gru.b_r, gru.b_z, gru.b_h])

print(f"LSTM parameters (D={D}, H={H}): {lstm_params}")
print(f"GRU  parameters (D={D}, H={H}): {gru_params}")
print(f"GRU uses {gru_params/lstm_params:.1%} of LSTM's parameters")

In [ ]:
# Standalone GRU comparison (no import needed)
D, H = 10, 16
# LSTM: 4 gates × (D+H+1) weight params per gate
lstm_params = 4 * H * (D + H) + 4 * H  # W + b
# GRU: 3 gates
gru_params = 3 * H * (D + H) + 3 * H

print(f"LSTM parameters (D={D}, H={H}): {lstm_params}")
print(f"GRU  parameters (D={D}, H={H}): {gru_params}")
print(f"GRU uses {gru_params/lstm_params:.1%} of LSTM's parameters")
print()

# Which is better? It depends on the task.
print("LSTM vs GRU empirically (Chung et al., 2014):")
print("  - Neither consistently outperforms the other")
print("  - GRU is faster (3/4 the compute)")
print("  - LSTM slightly better on very long sequences")
print("  - Default choice: GRU for efficiency, LSTM if you have compute")
print()
print("In 2024: Transformers have largely replaced both for NLP.")
print("But GRU/LSTM are still used in:")
print("  - Time series forecasting")
print("  - Streaming / low-latency inference")
print("  - Encoder for structured prediction tasks")

## Summary

**GRU simplification of LSTM:**
- No separate cell state — everything in hidden state
- Update gate = forget + input combined
- Reset gate = selective use of previous hidden state

**Gradient flow:** Same mechanism as LSTM — the (1-z) path carries gradient back with at most 1x multiplication, enabling longer memory.

---
**Next:** 05.4 — Seq2Seq with Encoder-Decoder